<a href="https://colab.research.google.com/github/nathalia2000-web/data-incident-copilot/blob/main/data_incident_copilot_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Incident Copilot

## Prueba de Concepto de Prompt Engineering para incidentes de calidad de datos

Autor: Natalia Puñal
Curso: IA. Ingenieria de Prompts
Comisión: 96090
  

---
El proyecto propone desarrollar una prueba de concepto de un copiloto para el análisis de incidentes de calidad de datos. La solución recibe información estructurada sobre un incidente —por ejemplo, tabla afectada, proceso involucrado, control de calidad, valor esperado, valor observado e impacto de negocio— y utiliza un modelo de inteligencia artificial generativa para producir un análisis inicial.

El copiloto busca asistir al equipo de datos en tareas como:

identificar y clasificar el tipo de incidente;
sugerir un nivel de severidad;
proponer posibles causas, expresadas como hipótesis;
generar consultas SQL de validación no destructivas;
recomendar acciones de remediación y prevención;
redactar un mensaje comprensible para áreas de negocio.
La solución se vincula con el desarrollo de modelos de IA porque utiliza un modelo de lenguaje de gran escala mediante API. No se entrena un modelo desde cero; en cambio, se aprovecha un modelo preentrenado y se controla su comportamiento mediante técnicas de prompt engineering. Esto permite construir una solución funcional en poco tiempo, con bajo costo de infraestructura y sin requerir un conjunto de datos etiquetado para entrenamiento.

Para evaluar el valor de las técnicas de prompting, se diseñan dos enfoques:

Prompt básico: presenta los datos principales del incidente y solicita un análisis general.
Prompt optimizado: define un rol especializado, reglas de seguridad, formato estructurado, delimitadores de contexto, hipótesis en lugar de afirmaciones no verificadas y un ejemplo breve de salida.
La comparación entre ambos prompts permite demostrar cómo una mejor instrucción al modelo puede aumentar la consistencia, trazabilidad y utilidad de las respuestas.


## Objetivo

Esta Prueba de Concepto utiliza Inteligencia Artificial Generativa para asistir en el análisis y la comunicación de incidentes de calidad de datos.

A partir de información estructurada sobre un incidente, la solución generará:

1. Un análisis técnico del incidente.
2. Una clasificación del tipo de problema y su severidad.
3. Hipótesis de causas posibles, sin presentarlas como hechos confirmados.
4. Consultas SQL genéricas para validar el problema.
5. Un plan de remediación y medidas preventivas.
6. Un mensaje comprensible para usuarios de negocio.

La POC no se conecta a bases de datos productivas, no ejecuta consultas SQL y no realiza cambios automáticos sobre los datos.


## Metodología

La POC se desarrollará mediante las siguientes etapas:

1. Cargar un conjunto de incidentes ficticios de calidad de datos desde un archivo CSV alojado en GitHub.
2. Seleccionar un incidente para analizar.
3. Construir un prompt básico, con una descripción simple del incidente.
4. Construir un prompt optimizado aplicando técnicas de Fast Prompting.
5. Consultar un modelo de IA de texto a texto con ambos prompts.
6. Comparar las respuestas obtenidas según claridad, estructura, utilidad técnica y cumplimiento de restricciones.
7. Generar, de manera opcional, una imagen ilustrativa para comunicar visualmente un incidente.
8. Analizar la cantidad de consultas realizadas para minimizar costos y evitar llamadas innecesarias a la API.

### Técnicas de Fast Prompting a utilizar

- **Role Prompting:** asignar al modelo el rol de ingeniero de datos especializado en calidad de datos.
- **Instrucciones explícitas:** indicar claramente las tareas que debe realizar.
- **Delimitadores:** separar la información del incidente de las instrucciones.
- **Salida estructurada:** solicitar campos definidos para facilitar la lectura y comparación.
- **Few-shot Prompting:** incorporar un ejemplo de entrada y salida esperada.
- **Restricciones:** pedir que las causas se expresen como hipótesis y no como hechos confirmados.
- **Descomposición de tareas:** dividir el análisis en clasificación, severidad, validación, remediación y comunicación.


In [19]:
# Importación de bibliotecas necesarias para leer y visualizar el archivo CSV.
import pandas as pd
from IPython.display import display

# Dirección RAW del archivo CSV almacenado en el repositorio de GitHub.
URL_DATASET = (
    "https://raw.githubusercontent.com/"
    "nathalia2000-web/"
    "data-incident-copilot/"
    "main/"
    "data/incidentes_ejemplo.csv"
)

# Lectura del archivo CSV desde GitHub.
df_incidentes = pd.read_csv(URL_DATASET)

# Visualización de los incidentes disponibles.
display(df_incidentes)


,id_incidente,fecha_deteccion,tabla_afectada,proceso,tipo_control,valor_esperado,valor_observado,impacto_negocio,descripcion
0,INC-001,2026-08-10,ventas_diarias,carga_etl_ventas,porcentaje_nulos_id_cliente,Menor a 1%,18%,El tablero de ventas por cliente presenta regi...,Aumento significativo de valores nulos en la c...
1,INC-002,2026-08-11,clientes_maestro,sync_crm_clientes,registros_duplicados,0 duplicados,245 duplicados,Posible duplicación de comunicaciones comerciales,Se detectaron registros duplicados según email...
2,INC-003,2026-08-12,inventario_diario,carga_api_inventario,volumen_registros,50000 registros,12300 registros,El reporte de stock podría mostrar información...,La cantidad de registros cargados es considera...
3,INC-004,2026-08-13,pagos_transacciones,etl_pagos_diarios,actualizacion_tabla,Actualización diaria antes de las 07:00,Última actualización hace 30 horas,Los indicadores financieros no están actualizados,La tabla no recibió registros durante la últim...
4,INC-005,2026-08-14,productos_catalogo,carga_catalogo_proveedores,cambio_de_esquema,Columna precio_unitario tipo decimal,Columna precio_unitario recibida como texto,La carga del catálogo de productos fue interru...,La fuente de origen modificó el tipo de dato d...
5,INC-006,2026-08-15,ordenes_entrega,etl_logistica,valores_fuera_de_rango,Estado dentro del catálogo permitido,Estado desconocido: ENTREGADO_PARCIALMENTE_X,Los reportes logísticos no clasifican correcta...,Se detectó un valor no contemplado en la colum...


In [20]:
# Selección del incidente que se utilizará para la demostración.
ID_INCIDENTE_SELECCIONADO = "INC-001"

# Se filtra el DataFrame para obtener únicamente el incidente seleccionado.
incidente_seleccionado = df_incidentes[
    df_incidentes["id_incidente"] == ID_INCIDENTE_SELECCIONADO
].iloc[0]

# Se visualiza la información del incidente elegido.
display(incidente_seleccionado.to_frame(name="valor"))


,valor
id_incidente,INC-001
fecha_deteccion,2026-08-10
tabla_afectada,ventas_diarias
proceso,carga_etl_ventas
tipo_control,porcentaje_nulos_id_cliente
valor_esperado,Menor a 1%
valor_observado,18%
impacto_negocio,El tablero de ventas por cliente presenta regi...
descripcion,Aumento significativo de valores nulos en la c...


In [21]:
# Construcción de un prompt básico.
# Este prompt contiene el incidente, pero tiene pocas instrucciones sobre
# el rol del modelo, las restricciones y el formato esperado.

prompt_basico = f"""
Analiza el siguiente incidente de calidad de datos y propone una solución.

Tabla afectada: {incidente_seleccionado["tabla_afectada"]}
Proceso: {incidente_seleccionado["proceso"]}
Problema: {incidente_seleccionado["descripcion"]}
Valor esperado: {incidente_seleccionado["valor_esperado"]}
Valor observado: {incidente_seleccionado["valor_observado"]}
Impacto de negocio: {incidente_seleccionado["impacto_negocio"]}
"""

print(prompt_basico)



Analiza el siguiente incidente de calidad de datos y propone una solución.

Tabla afectada: ventas_diarias
Proceso: carga_etl_ventas
Problema: Aumento significativo de valores nulos en la columna id_cliente.
Valor esperado: Menor a 1%
Valor observado: 18%
Impacto de negocio: El tablero de ventas por cliente presenta registros incompletos



## Selección del incidente y construcción del prompt básico

Para la demostración se seleccionó el incidente `INC-001`, asociado a un aumento de valores nulos en la columna `id_cliente` de la tabla `ventas_diarias`.

En esta etapa se construye un prompt básico que contiene únicamente los datos principales del incidente y una instrucción general para analizarlo.

Este prompt servirá como línea de base para comparar posteriormente su resultado con un prompt optimizado mediante técnicas de Fast Prompting.


## Construcción del prompt optimizado

A diferencia del prompt básico, el siguiente prompt incorpora técnicas de Fast Prompting para orientar al modelo hacia una respuesta más consistente, segura y reutilizable.

Las principales mejoras aplicadas son:

- Se asigna un rol especializado al modelo.
- Se define una tarea concreta y descompuesta en secciones.
- Se delimita la información del incidente para evitar confusiones entre datos e instrucciones.
- Se solicita que las causas posibles se expresen como hipótesis.
- Se incluye un ejemplo breve de clasificación.
- Se establece una estructura de salida definida.
- Se solicita un mensaje diferenciado para usuarios de negocio.


In [22]:
import json

# Conversión del incidente seleccionado a formato JSON.
# Esto permite delimitar y presentar los datos de forma clara al modelo.
incidente_json = json.dumps(
    incidente_seleccionado.to_dict(),
    ensure_ascii=False,
    indent=2
)

# Construcción del prompt optimizado aplicando técnicas de Fast Prompting.
prompt_optimizado = f"""
# Rol

Eres un ingeniero de datos especializado en calidad, observabilidad y
gestión de incidentes de datos.

# Objetivo

Analiza el incidente proporcionado y genera una respuesta útil para un
equipo técnico y para usuarios de negocio.

# Reglas obligatorias

1. Considera las causas posibles como hipótesis, no como hechos confirmados.
2. No afirmes que ejecutaste consultas SQL, revisaste sistemas o accediste
   a bases de datos.
3. Propón consultas SQL genéricas de validación; no incluyas comandos que
   modifiquen, eliminen o actualicen datos.
4. Usa lenguaje claro, preciso y profesional.
5. Si falta información, indícalo dentro de las hipótesis o recomendaciones.
6. Basa el nivel de severidad en el impacto de negocio y la diferencia entre
   el valor esperado y el valor observado.

# Ejemplo breve

Entrada:
- Tipo de control: registros_duplicados
- Valor esperado: 0 duplicados
- Valor observado: 100 duplicados

Resultado esperado:
- Clasificación: Duplicidad de registros.
- Severidad sugerida: Media o Alta, según el impacto de negocio.
- Causa posible: Hipótesis de ausencia o falla en una regla de deduplicación.

# Incidente a analizar

<incidente>
{incidente_json}
</incidente>

# Formato obligatorio de salida

Responde únicamente con un objeto JSON válido que contenga estas claves:

{{
  "resumen_tecnico": "Descripción breve del problema detectado.",
  "clasificacion": "Tipo de incidente de calidad de datos.",
  "severidad_sugerida": "Baja, Media, Alta o Crítica.",
  "justificacion_severidad": "Motivo de la severidad propuesta.",
  "hipotesis_de_causa": [
    "Hipótesis 1.",
    "Hipótesis 2.",
    "Hipótesis 3."
  ],
  "consultas_sql_de_validacion": [
    "Consulta SQL genérica 1.",
    "Consulta SQL genérica 2."
  ],
  "plan_de_remediacion": [
    "Acción inmediata 1.",
    "Acción inmediata 2.",
    "Acción inmediata 3."
  ],
  "medidas_preventivas": [
    "Medida preventiva 1.",
    "Medida preventiva 2."
  ],
  "mensaje_para_negocio": "Mensaje claro y no técnico para las áreas afectadas."
}}
"""

# Visualización del prompt optimizado antes de enviarlo a un modelo.
print(prompt_optimizado)



# Rol

Eres un ingeniero de datos especializado en calidad, observabilidad y
gestión de incidentes de datos.

# Objetivo

Analiza el incidente proporcionado y genera una respuesta útil para un
equipo técnico y para usuarios de negocio.

# Reglas obligatorias

1. Considera las causas posibles como hipótesis, no como hechos confirmados.
2. No afirmes que ejecutaste consultas SQL, revisaste sistemas o accediste
   a bases de datos.
3. Propón consultas SQL genéricas de validación; no incluyas comandos que
   modifiquen, eliminen o actualicen datos.
4. Usa lenguaje claro, preciso y profesional.
5. Si falta información, indícalo dentro de las hipótesis o recomendaciones.
6. Basa el nivel de severidad en el impacto de negocio y la diferencia entre
   el valor esperado y el valor observado.

# Ejemplo breve

Entrada:
- Tipo de control: registros_duplicados
- Valor esperado: 0 duplicados
- Valor observado: 100 duplicados

Resultado esperado:
- Clasificación: Duplicidad de registros.
- Severida

Reglas obligatorias de generación:

- Devuelve únicamente un objeto JSON válido.
- No agregues explicaciones, títulos ni texto fuera del JSON.
- Respeta exactamente los nombres de las claves indicadas.
- Utiliza comillas dobles para todas las claves y valores de texto.
- No utilices bloques Markdown ni ```json.
- Sé conciso:
  - resumen_tecnico: máximo 3 oraciones.
  - justificacion_severidad: máximo 2 oraciones.
  - hipotesis_de_causa: máximo 3 elementos.
  - consultas_sql_de_validacion: máximo 3 elementos.
  - plan_de_remediacion: máximo 4 elementos.
  - medidas_preventivas: máximo 4 elementos.
  - mensaje_para_negocio: máximo 2 oraciones.

Reglas para las consultas SQL:

- Genera únicamente consultas SELECT de lectura.
- Utiliza COUNT(*) y nunca COUNT().
- Verifica que todos los operadores aritméticos estén presentes.
- No generes UPDATE, DELETE, INSERT, DROP ni ALTER.
- Las consultas deben ser sintácticamente válidas.
- Si calculas porcentajes, utiliza una expresión equivalente a:
  (
      SUM(CASE WHEN columna IS NULL THEN 1 ELSE 0 END)::numeric
      / COUNT(*)::numeric
  ) * 100
- Revisa las consultas SQL antes de devolver el JSON.


## Comparación entre prompt básico y prompt optimizado

| Criterio | Prompt básico | Prompt optimizado |
|---|---|---|
| Rol definido | No | Sí: ingeniero de datos especializado en calidad y observabilidad |
| Contexto del incidente | Información textual simple | Información delimitada en formato JSON |
| Instrucciones de análisis | General: analizar y proponer una solución | Específicas: clasificar, asignar severidad, formular hipótesis y proponer acciones |
| Control de afirmaciones no verificadas | No | Sí: las causas deben formularse como hipótesis |
| Seguridad de consultas SQL | No especificada | Solo se permiten consultas de validación, sin modificar datos |
| Ejemplo de salida | No | Sí: ejemplo few-shot de clasificación de duplicados |
| Formato de respuesta | Libre | JSON estructurado con campos obligatorios |
| Público objetivo | No diferenciado | Incluye análisis técnico y mensaje para negocio |
| Reutilización y automatización | Limitada | Alta, por su estructura consistente |

El prompt básico puede producir respuestas variables, poco estructuradas o con recomendaciones incompletas. El prompt optimizado reduce esa ambigüedad al definir el rol, el contexto, las restricciones y el formato de salida esperado.

La integración con la API fue configurada utilizando Google Colab Secrets para no exponer la clave en el repositorio. Debido a la falta de créditos disponibles, no se realizaron reintentos automáticos, como medida de control de costos.


Justificación de la viabilidad del proyecto
El proyecto es técnicamente viable porque utiliza herramientas accesibles, ampliamente documentadas y compatibles entre sí:

GitHub permite almacenar y versionar el dataset CSV y el notebook.
Google Colab proporciona un entorno de desarrollo en la nube sin requerir instalación local.
Python y pandas permiten leer, filtrar y transformar los incidentes del dataset.
La biblioteca Groq permite integrar un modelo de lenguaje mediante API.
Google Colab Secrets permite almacenar la clave API fuera del código fuente.
La propuesta es viable dentro del tiempo disponible porque no requiere entrenar, ajustar ni desplegar un modelo propio. El esfuerzo se concentra en:

estructurar correctamente los datos de entrada;
diseñar prompts claros y seguros;
implementar la integración básica;
documentar resultados, limitaciones y decisiones técnicas.
Además, el proyecto está diseñado como una POC. Esto reduce su alcance frente a un sistema productivo: no incluye integración directa con bases de datos corporativas, autenticación de usuarios, monitoreo continuo, ejecución automática de SQL ni mecanismos de aprobación humana. Estas funcionalidades pueden considerarse etapas futuras.



Herramientas y tecnologías

Herramientas utilizadas

Herramienta o tecnologia:

Google Colab: Desarrollo y ejecución del notebook en la nube

Python: Lenguaje principal de implementación

pandas: Lectura, filtrado y visualización del dataset CSV

GitHub: Almacenamiento, versionado y acceso al dataset/notebook

Groq Python SDK: Integración con el modelo de IA mediante API

Google Colab Secrets: Almacenamiento seguro de la clave API

JSON: Formato estructurado para el incidente y la respuesta esperada

Técnicas de prompting utilizadas
1. Definición de rol
El prompt optimizado asigna al modelo el rol de:

Ingeniero de datos especializado en calidad, observabilidad y gestión de incidentes de datos.

Esta técnica orienta el tipo de razonamiento, vocabulario y recomendaciones esperadas.

2. Contexto estructurado
Los datos del incidente se convierten a JSON y se colocan entre etiquetas:


<incidente>
...
</incidente>

Esto separa claramente las instrucciones de los datos que deben ser analizados. Además, facilita que el modelo identifique los campos disponibles.

3. Instrucciones explícitas
El prompt especifica las tareas esperadas: clasificar el incidente, sugerir severidad, formular hipótesis, proponer SQL de validación, crear un plan de remediación y redactar un mensaje para negocio.

Esta técnica disminuye la ambigüedad presente en una instrucción general como “analiza el incidente”.

4. Restricciones de seguridad y confiabilidad
Se incluyen reglas para evitar comportamientos no deseados:

No afirmar que se ejecutaron consultas o se accedió a sistemas;
Presentar las causas como hipótesis;
No proponer SQL que modifique, elimine o actualice datos;
Indicar cuando falta información.
Estas restricciones son relevantes porque un modelo generativo puede producir contenido convincente aunque no esté validado. El objetivo es que la respuesta sea útil sin presentarse como evidencia definitiva.

5. Few-shot prompting
El prompt incorpora un ejemplo breve de entrada y salida esperada para un caso de registros duplicados.

Este ejemplo ayuda al modelo a interpretar la estructura deseada y a mantener un criterio coherente de clasificación y severidad.

6. Salida estructurada
Se solicita una respuesta exclusivamente en formato JSON, con campos definidos previamente.

Esta técnica favorece la consistencia y permitiría, en una futura versión, integrar la respuesta en tableros, tickets de incidentes, sistemas de alerta o flujos automáticos de revisión humana.

La propuesta demuestra que un modelo de IA generativa puede utilizarse como asistente inicial para la gestión de incidentes de calidad de datos. El valor del proyecto no depende únicamente del modelo utilizado, sino de la calidad de las instrucciones diseñadas.

El prompt optimizado mejora al prompt básico porque incorpora un rol especializado, restricciones de seguridad, contexto estructurado, un formato de salida definido y una diferenciación entre necesidades técnicas y de negocio. Estas características aumentan la posibilidad de reutilizar la solución en futuras automatizaciones, manteniendo siempre la necesidad de validación humana antes de tomar decisiones operativas.



In [34]:
%pip install -q groq

In [35]:
from groq import Groq
print( "Biblioteca de groq instalada correctamente")

Biblioteca de groq instalada correctamente


In [40]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError(
        "No se encontró GROQ_API_KEY. "
        "Verifica que el secreto exista y que tenga activado "
        "el acceso desde el notebook."
    )

client = Groq(api_key=GROQ_API_KEY)

MODEL_NAME = "openai/gpt-oss-20b"

print("Cliente de Groq configurado correctamente.")
print(f"Modelo seleccionado: {MODEL_NAME}")

Cliente de Groq configurado correctamente.
Modelo seleccionado: openai/gpt-oss-20b


In [45]:
def consultar_groq(prompt, salida_json=False):
    parametros = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": str(prompt)
            }
        ],
        "temperature": 0.2,
        "max_tokens": 4096
    }

    if salida_json:
        parametros["response_format"] = {
            "type": "json_object"
        }

    try:
        respuesta = client.chat.completions.create(**parametros)
        return respuesta.choices[0].message.content

    except Exception as error:
        print("ERROR AL CONSULTAR GROQ")
        print("Tipo:", type(error).__name__)
        print("Detalle:", error)
        return None

In [42]:
prueba = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Responde únicamente: conexión correcta"
        }
    ],
    temperature=0.2,
    max_tokens=50
)

print(prueba.choices[0].message.content)

conex


In [43]:
respuesta_basica = consultar_groq(prompt_basico)

print("RESPUESTA DEL PROMPT BÁSICO\n")
print(respuesta_basica)

RESPUESTA DEL PROMPT BÁSICO

## 1. Resumen del Incidente  
| Elemento | Detalle |
|----------|---------|
| **Tabla afectada** | `ventas_diarias` |
| **Proceso** | `carga_etl_ventas` |
| **Problema** | Aumento de valores nulos en la columna `id_cliente` |
| **Valor esperado** | < 1 % de nulos |
| **Valor observado** | 18 % de nulos |
| **Impacto** | El tablero de ventas por cliente muestra registros incompletos y distorsiona los análisis de rendimiento y segmentación. |

---

## 2. Diagnóstico (Root‑Cause Analysis)

| Posible causa | Evidencia | Comentario |
|---------------|-----------|------------|
| **1. Cambios en la fuente de datos** | Se detectó que la tabla origen (p.ej. `clientes_raw`) empezó a enviar registros sin `id_cliente` después de la actualización de la aplicación de facturación. | La fuente no valida la integridad referencial. |
| **2. Transformación ETL defectuosa** | El script `carga_etl_ventas` utiliza una unión externa (`LEFT JOIN`) con `clientes_raw` sin filtrar lo

In [46]:
respuesta_optimizada = consultar_groq(
    prompt_optimizado,
    salida_json=True
)

if respuesta_optimizada is not None:
    print("RESPUESTA DEL PROMPT OPTIMIZADO\n")
    print(respuesta_optimizada)

RESPUESTA DEL PROMPT OPTIMIZADO

{"resumen_tecnico":"Se detectó un alto porcentaje de valores nulos (18%) en la columna id_cliente de la tabla ventas_diarias, lo que afecta la integridad de los datos de clientes en el tablero de ventas.","clasificacion":"Datos faltantes en columna crítica","severidad_sugerida":"Alta","justificacion_severidad":"El 18% de registros con id_cliente nulo impide la correcta agregación y análisis por cliente, afectando decisiones de negocio.","hipotesis_de_causa":["Fallo en la transformación ETL que no valida o rellena id_cliente antes de cargar.","Fuente de datos original contiene valores nulos o campos vacíos en la columna id_cliente.","Problema de mapeo de columnas durante la carga, donde la columna id_cliente se asigna incorrectamente o se omite."],"consultas_sql_de_validacion":["SELECT COUNT(*) AS total, SUM(CASE WHEN id_cliente IS NULL THEN 1 ELSE 0 END) AS null_count, (SUM(CASE WHEN id_cliente IS NULL THEN 1 ELSE 0 END)::numeric / COUNT(*)::numeric)*10

In [47]:
import json

try:
    resultado_optimizado = json.loads(respuesta_optimizada)

    print("La respuesta optimizada es un JSON válido.")
    print("Claves recibidas:")
    print(list(resultado_optimizado.keys()))

except json.JSONDecodeError:
    resultado_optimizado = None
    print("La respuesta optimizada no tiene un formato JSON válido.")

La respuesta optimizada es un JSON válido.
Claves recibidas:
['resumen_tecnico', 'clasificacion', 'severidad_sugerida', 'justificacion_severidad', 'hipotesis_de_causa', 'consultas_sql_de_validacion', 'plan_de_remediacion', 'medidas_preventivas', 'mensaje_para_negocio']


In [48]:
claves_obligatorias = {
    "resumen_tecnico",
    "clasificacion",
    "severidad_sugerida",
    "justificacion_severidad",
    "hipotesis_de_causa",
    "consultas_sql_de_validacion",
    "plan_de_remediacion",
    "medidas_preventivas",
    "mensaje_para_negocio"
}

if resultado_optimizado is not None:
    faltantes = claves_obligatorias - set(resultado_optimizado.keys())

    if not faltantes:
        print("La respuesta contiene todas las claves obligatorias.")
    else:
        print("Faltan las siguientes claves:")
        print(faltantes)

La respuesta contiene todas las claves obligatorias.


## Validación de la respuesta
Se verificó que la respuesta generada por el modelo tuviera un formato JSON válido. También se comprobó que incluyera todas las claves definidas en el esquema de salida: resumen técnico, clasificación, severidad sugerida, justificación de severidad, hipótesis de causa, consultas SQL de validación, plan de remediación, medidas preventivas y mensaje para negocio.

Además, se realizó una revisión manual de las consultas SQL generadas para comprobar que fueran de tipo SELECT y que utilizaran una sintaxis adecuada. Esta revisión es necesaria porque las respuestas generadas por un modelo de lenguaje deben validarse antes de utilizarse en un entorno real.

In [49]:
import json

try:
    resultado_optimizado = json.loads(respuesta_optimizada)

    print("La respuesta optimizada es un JSON válido.")
    print("Claves recibidas:")
    print(list(resultado_optimizado.keys()))

except json.JSONDecodeError as error:
    resultado_optimizado = None
    print("La respuesta no es un JSON válido.")
    print("Detalle del error:", error)

La respuesta optimizada es un JSON válido.
Claves recibidas:
['resumen_tecnico', 'clasificacion', 'severidad_sugerida', 'justificacion_severidad', 'hipotesis_de_causa', 'consultas_sql_de_validacion', 'plan_de_remediacion', 'medidas_preventivas', 'mensaje_para_negocio']


In [50]:
claves_obligatorias = {
    "resumen_tecnico",
    "clasificacion",
    "severidad_sugerida",
    "justificacion_severidad",
    "hipotesis_de_causa",
    "consultas_sql_de_validacion",
    "plan_de_remediacion",
    "medidas_preventivas",
    "mensaje_para_negocio"
}

if resultado_optimizado is not None:
    faltantes = claves_obligatorias - set(resultado_optimizado.keys())

    if not faltantes:
        print("La respuesta contiene todas las claves obligatorias.")
    else:
        print("Faltan las siguientes claves:")
        print(faltantes)

La respuesta contiene todas las claves obligatorias.


### Resultado de la validación

La respuesta fue interpretada correctamente como un objeto JSON y contenía todas las claves obligatorias. Esto confirma que el modelo respetó la estructura solicitada en el prompt optimizado.

Las consultas SQL fueron revisadas manualmente. Cuando fue necesario, se corrigieron detalles sintácticos como el uso de COUNT(*) y la inclusión del operador de multiplicación en el cálculo del porcentaje de valores nulos.

## Resultados


## Modelo texto-imagen

## Conclusiones